In [ ]:
# | Type                            | Definition                                                                                                                         |
# | ------------------------------- | ---------------------------------------------------------------------------------------------------------------------------------- |
# | **Pydantic Structured Output**  | Converts LLM responses into a predefined Python object with automatic type checking and validation using Pydantic models.          |
# | **TypedDict Structured Output** | TypeDict convert LLM output to expected python dictionary with lightweight type hints but without validation. |
# | **JSON Structured Output**      | Forces the LLM to return data in a valid JSON format that can be easily used by programs or APIs.                                  |
# | **StrOutputParser**             | Converts the LLM response into a simple string/text format without any complex structure.                                          |
# | **StructuredOutputParser**      | It Parses LLM responses into a custom predefined structure using specified fields and descriptions.                                   |


In [1]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()
from langchain_core.prompts import PromptTemplate

/Users/rachin/Desktop/GenAI/genai/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = ChatOpenAI(model = "gpt-4o-mini")

# Stroutput parser

In [5]:
from langchain_core.output_parsers import StrOutputParser

# prompt 1 template for  generate a detailed report on a given topic
template1 = PromptTemplate(template= "write a detailed report on given {topic} topic", input_variables= ["topic"])

# prompt template 2 for generating a summary based on the generated report
template2 = PromptTemplate(template= "write a 4 lines summaries basaed on the given input \n {input}", input_variables= ["input"])

parser = StrOutputParser()

chain = template1 | model | parser | template2 | model | parser

response = chain.invoke({"topic":"Universe"})

print(response)

The universe is an expansive realm encompassing all celestial objects and governed by physical laws, with its origin traced back to the Big Bang around 13.8 billion years ago. Evidence like Cosmic Microwave Background Radiation supports this theory, while the universe comprises various galaxies, dark matter, and dark energy, constituting its complex composition. Current expansion trends suggest possible futures, including scenarios like the Big Freeze and the Big Rip, shaping the ongoing discourse in cosmology. Continued research aims to unravel the universe's mysteries, enhancing our understanding of its structure and our place within it.


## Pydantic output parser

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

class Person(BaseModel):
    name: str = Field(description='Name of the person')
    age: int = Field(gt=18, description='Age of the person')
    city: str = Field(description='Name of the city the person belongs to')



parser = PydanticOutputParser(pydantic_object= Person)


In [10]:
template3 = PromptTemplate(
    template = 'Generate the name, age and city of a fictional {place} person \n {format_instruction}',
    input_variables= ["place"],
    partial_variables= {"format_instruction": parser.get_format_instructions()}
)

chain = template3 | model | parser

result = chain.invoke({"place":"bangladesh"})

print(result)

name='Rafiq Ahmed' age=25 city='Dhaka'


## Structured Output Parser

In [11]:
from langchain_classic.output_parsers.structured import StructuredOutputParser, ResponseSchema

schema = [
    ResponseSchema(name ="fact_1", description= "Fact 1 about the topic"),
    ResponseSchema(name ="fact_2", description= "Fact 2 about the topic"),
    ResponseSchema(name ="fact_3", description= "Fact 3 about the topic"),
]
parser = StructuredOutputParser.from_response_schemas(schema)

template = PromptTemplate(
    template='Give 3 fact about {topic} \n {format_instruction}',
    input_variables=['topic'],
    partial_variables={'format_instruction':parser.get_format_instructions()}
)

chain = template | model | parser

result = chain.invoke({'topic':'black hole'})

print(result)


{'fact_1': 'Black holes are regions in space where gravity is so strong that nothing, not even light, can escape from them.', 'fact_2': 'The boundary surrounding a black hole is called the event horizon; once something crosses this boundary, it cannot return.', 'fact_3': 'Black holes can be formed from the remnants of massive stars that collapse under their own gravity after exhausting their nuclear fuel.'}
